# 2단계 LightGBM

In [11]:
from pathlib import Path

CUR_DIR = Path.cwd().resolve()
BASE_DIR = CUR_DIR / "data"

In [12]:
import os
import gc
import numpy as np
import pandas as pd
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, brier_score_loss
)
from scipy.stats import spearmanr


DRIVE_DIR = BASE_DIR
PARQUET_PATH = os.path.join(DRIVE_DIR, '2단계_학습데이터_최종_인허가포함_라벨보정.parquet')

CHECKPOINT_DIR = os.path.join(
    CUR_DIR,
    "result",
    "step2_model"
)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
ROUND_METRICS_PATH = os.path.join(CHECKPOINT_DIR, '2단계_lgbm_v6_walkforward_지표.csv')
OOF_PRED_PATH = os.path.join(CHECKPOINT_DIR, '2단계_lgbm_v6_OOF예측.csv')

ORIGINAL_TARGET_COL = '폐업_다음분기'
TARGET_COL = '최종_보정라벨'   
QUARTER_COL = '기준_년분기'
ID_COL = '상가업소번호'
MIN_TRAIN_QUARTERS = 8

FIXED_PARAMS = dict(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=6,
    num_leaves=31,
    min_child_samples=100,
    class_weight='balanced',
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)


## 1. 데이터 로드

In [13]:
df = pd.read_parquet(PARQUET_PATH)
df = df.sort_values([ID_COL, QUARTER_COL]).reset_index(drop=True)
print(f'✅ Shape: {df.shape}')


✅ Shape: (6041091, 60)


## 2. 라벨 보정

In [14]:
df['최종_보정라벨'] = df[ORIGINAL_TARGET_COL]

matched = df['인허가매칭성공여부'] == 1

false_positive = (
    matched
    & (df['라벨_비교결과'] == '불일치_기존만폐업')
    & df['폐업일자'].isna()
    & (df['영업상태명'] == '영업/정상')
)
df.loc[false_positive, '최종_보정라벨'] = 0

false_negative = matched & (df['라벨_비교결과'] == '불일치_인허가만폐업')
df.loc[false_negative, '최종_보정라벨'] = 1

n_changed = (df['최종_보정라벨'] != df[ORIGINAL_TARGET_COL]).sum()
print(f'보정된 행: {n_changed:,}건 (1→0: {false_positive.sum():,} / 0→1: {false_negative.sum():,})')

df['최종_보정라벨'] = df['최종_보정라벨'].astype('int8')


보정된 행: 26,062건 (1→0: 18,365 / 0→1: 7,697)


## 3. 갭 안전 헬퍼 함수

In [15]:
def quarter_add(q, delta):
    year = int(str(q)[:4]); qtr = int(str(q)[-1])
    total = year * 4 + (qtr - 1) + delta
    return f'{total // 4}Q{total % 4 + 1}'


def add_lagged_lookup(data, id_col, quarter_col, cols, lag):
    lookup = data[[id_col, quarter_col] + cols].rename(
        columns={quarter_col: '_lag_q', **{c: f'{c}_lag{lag}' for c in cols}}
    )
    key = pd.DataFrame({
        id_col: data[id_col].values,
        '_lag_q': data[quarter_col].map(lambda q: quarter_add(q, -lag)).values,
    })
    result = key.merge(lookup, on=[id_col, '_lag_q'], how='left')
    return result[[f'{c}_lag{lag}' for c in cols]].reset_index(drop=True)


def add_expanding_target_rate(data, group_col, quarter_col, target_col, out_col, smoothing_k=50):
    agg = data.groupby([group_col, quarter_col], observed=True)[target_col].agg(sum_='sum', count_='count').reset_index()
    agg = agg.sort_values([group_col, quarter_col])
    agg['cum_sum'] = agg.groupby(group_col, observed=True)['sum_'].cumsum() - agg['sum_']
    agg['cum_count'] = agg.groupby(group_col, observed=True)['count_'].cumsum() - agg['count_']

    global_mean = data[target_col].mean()
    agg[out_col] = (agg['cum_sum'] + smoothing_k * global_mean) / (agg['cum_count'] + smoothing_k)

    result = data[[group_col, quarter_col]].merge(
        agg[[group_col, quarter_col, out_col]], on=[group_col, quarter_col], how='left'
    )
    return result[out_col].to_numpy()


## 4. 신규 피처 생성

In [16]:
NEW_FEATURES = []

if '상권_코드' in df.columns:
    df['상권_누적_폐업률'] = add_expanding_target_rate(df, '상권_코드', QUARTER_COL, TARGET_COL, '상권_누적_폐업률', smoothing_k=50)
    NEW_FEATURES.append('상권_누적_폐업률')

if '서비스_업종_코드' in df.columns:
    df['업종_누적_폐업률'] = add_expanding_target_rate(df, '서비스_업종_코드', QUARTER_COL, TARGET_COL, '업종_누적_폐업률', smoothing_k=50)
    NEW_FEATURES.append('업종_누적_폐업률')

DIFF4_COLS = ['1단계_위험점수', '반경100m_이웃수']
lag4 = add_lagged_lookup(df, ID_COL, QUARTER_COL, DIFF4_COLS, lag=4)
for c in DIFF4_COLS:
    df[f'{c}_diff4'] = (df[c].to_numpy() - lag4[f'{c}_lag4'].to_numpy()).astype('float32')
    NEW_FEATURES.append(f'{c}_diff4')
del lag4

lag1 = add_lagged_lookup(df, ID_COL, QUARTER_COL, ['1단계_위험점수'], lag=1)
lag2 = add_lagged_lookup(df, ID_COL, QUARTER_COL, ['1단계_위험점수'], lag=2)
lag3 = add_lagged_lookup(df, ID_COL, QUARTER_COL, ['1단계_위험점수'], lag=3)
lag4b = add_lagged_lookup(df, ID_COL, QUARTER_COL, ['1단계_위험점수'], lag=4)
stacked = np.vstack([
    lag1['1단계_위험점수_lag1'].to_numpy(), lag2['1단계_위험점수_lag2'].to_numpy(),
    lag3['1단계_위험점수_lag3'].to_numpy(), lag4b['1단계_위험점수_lag4'].to_numpy(),
])
with np.errstate(invalid='ignore'):
    df['1단계_위험점수_roll_std_4q'] = np.nanstd(stacked, axis=0).astype('float32')
NEW_FEATURES.append('1단계_위험점수_roll_std_4q')
del lag1, lag2, lag3, lag4b, stacked
gc.collect()

qtr_num = df[QUARTER_COL].str[-1].astype(int)
for q in [2, 3, 4]:
    col = f'분기_Q{q}여부'
    df[col] = (qtr_num == q).astype('int8')
    NEW_FEATURES.append(col)

if '상권_누적_폐업률' in df.columns and '업종_누적_폐업률' in df.columns:
    df['상권_대비_업종_폐업률_차이'] = (df['상권_누적_폐업률'] - df['업종_누적_폐업률']).astype('float32')
    NEW_FEATURES.append('상권_대비_업종_폐업률_차이')

lag1n = add_lagged_lookup(df, ID_COL, QUARTER_COL, ['반경100m_이웃수'], lag=1)
lag2n = add_lagged_lookup(df, ID_COL, QUARTER_COL, ['반경100m_이웃수'], lag=2)
lag3n = add_lagged_lookup(df, ID_COL, QUARTER_COL, ['반경100m_이웃수'], lag=3)
lag4n = add_lagged_lookup(df, ID_COL, QUARTER_COL, ['반경100m_이웃수'], lag=4)
stacked_n = np.vstack([
    lag1n['반경100m_이웃수_lag1'].to_numpy(), lag2n['반경100m_이웃수_lag2'].to_numpy(),
    lag3n['반경100m_이웃수_lag3'].to_numpy(), lag4n['반경100m_이웃수_lag4'].to_numpy(),
])
with np.errstate(invalid='ignore'):
    roll_mean_4q = np.nanmean(stacked_n, axis=0)
df['반경100m_이웃수_대비_4분기평균비율'] = (df['반경100m_이웃수'].to_numpy() / (roll_mean_4q + 1.0)).astype('float32')
NEW_FEATURES.append('반경100m_이웃수_대비_4분기평균비율')
del lag1n, lag2n, lag3n, lag4n, stacked_n, roll_mean_4q
gc.collect()

print(f'✅ 신규 피처 {len(NEW_FEATURES)}개:', NEW_FEATURES)


✅ 신규 피처 10개: ['상권_누적_폐업률', '업종_누적_폐업률', '1단계_위험점수_diff4', '반경100m_이웃수_diff4', '1단계_위험점수_roll_std_4q', '분기_Q2여부', '분기_Q3여부', '분기_Q4여부', '상권_대비_업종_폐업률_차이', '반경100m_이웃수_대비_4분기평균비율']


## 5. 최종 FEATURE_COLS

In [17]:
META_COLS = ['상권_코드', '상권_코드_명', '서비스_업종_코드', '서비스_업종_코드_명']
META_COLS_LICENSE = ['관리번호', '매칭등급', '인허가일자', '폐업일자', '영업상태명',
                      '라벨_비교결과', '인허가기반_폐업_다음분기']
EXCLUDE_COLS = [ORIGINAL_TARGET_COL, TARGET_COL, '원본_폐업_다음분기', QUARTER_COL, ID_COL] + META_COLS + META_COLS_LICENSE
LOW_VALUE_FEATURES = ['매칭등급_B', '매칭등급_C', '매칭등급_D', '업력관측여부', '인허가매칭성공여부']

FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE_COLS + LOW_VALUE_FEATURES]
print(f'✅ 최종 피처 개수: {len(FEATURE_COLS)}개')


✅ 최종 피처 개수: 49개


In [18]:
from pathlib import Path
import gc

CUR_DIR = Path.cwd().resolve()
BASE_DIR = CUR_DIR / "data"

FEATURE_ENGINEERED_PATH = (
    BASE_DIR
    / "2단계_학습데이터_최종_인허가포함_피처완료.parquet"
)

required_cols = [
    "최종_보정라벨",
    "기준_년분기",
    "상가업소번호",
]

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(
        f"피처 완료 데이터에 필수 컬럼이 없습니다: {missing_cols}"
    )

df.to_parquet(
    FEATURE_ENGINEERED_PATH,
    index=False,
    engine="pyarrow",
)

gc.collect()

print("피처 완료 파일 저장 완료")
print("경로:", FEATURE_ENGINEERED_PATH)
print("크기:", df.shape)

피처 완료 파일 저장 완료
경로: .\data\2단계_학습데이터_최종_인허가포함_피처완료.parquet
크기: (6041091, 70)


## 6. Walk-Forward

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
ALL_QUARTERS = sorted(df[QUARTER_COL].unique())

def generate_walkforward_splits(all_quarters, min_train_q):
    for i in range(min_train_q, len(all_quarters)):
        yield all_quarters[:i], all_quarters[i]

def find_best_f1(y_true, y_pred_proba):
    best_thresh, best_f1 = 0.5, 0
    for thresh in np.arange(0.05, 0.85, 0.02):
        pred = (y_pred_proba >= thresh).astype(int)
        score = f1_score(y_true, pred, zero_division=0)
        if score > best_f1:
            best_f1, best_thresh = score, thresh
    return best_thresh, best_f1


round_results = []
oof_predictions = []

for round_idx, (train_q, val_q) in enumerate(generate_walkforward_splits(ALL_QUARTERS, MIN_TRAIN_QUARTERS)):
    print(f'\n{"-"*50}')
    print(f'🔄 [Round {round_idx + 1}] Train: {train_q[0]} ~ {train_q[-1]} | Val: {val_q}')

    train_df = df[df[QUARTER_COL].isin(train_q)]
    val_df = df[df[QUARTER_COL] == val_q]

    X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET_COL].values
    X_val, y_val = val_df[FEATURE_COLS], val_df[TARGET_COL].values

    model = LGBMClassifier(**FIXED_PARAMS)
    model.fit(X_train, y_train)

    _importance = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
    _top_share = _importance.iloc[0] / _importance.sum()
    if _top_share > 0.5:
        print(f"   ⚠️ 경고: 상위 피처('{_importance.index[0]}')가 전체 importance의 "
              f'{_top_share*100:.1f}%를 차지함 - 누수 의심, 결과 신뢰하지 말고 먼저 확인 필요')
    print('   상위 5개 피처:', dict(_importance.head(5)))

    inner_val_q = train_q[-1]
    inner_val_df = df[df[QUARTER_COL] == inner_val_q]
    X_inner_val, y_inner_val = inner_val_df[FEATURE_COLS], inner_val_df[TARGET_COL].values

    calibrator = CalibratedClassifierCV(
    model,
    method='isotonic',
    cv='prefit'
)
    calibrator.fit(X_inner_val, y_inner_val)

    y_pred_proba_raw = model.predict_proba(X_val)[:, 1]
    y_pred_proba = calibrator.predict_proba(X_val)[:, 1]

    best_thresh, best_f1 = find_best_f1(y_val, y_pred_proba)
    y_pred_binary = (y_pred_proba >= best_thresh).astype(int)

    auc = roc_auc_score(y_val, y_pred_proba)
    pr_auc = average_precision_score(y_val, y_pred_proba)
    prec = precision_score(y_val, y_pred_binary, zero_division=0)
    rec = recall_score(y_val, y_pred_binary, zero_division=0)
    spearman, _ = spearmanr(y_pred_proba, y_val)
    brier_raw = brier_score_loss(y_val, y_pred_proba_raw)
    brier_calibrated = brier_score_loss(y_val, y_pred_proba)

    print(f'📊 ROC-AUC: {auc:.4f} | PR-AUC: {pr_auc:.4f} | Spearman: {spearman:.4f}')
    print(f'   Brier (raw->calibrated): {brier_raw:.4f} -> {brier_calibrated:.4f}')

    round_results.append({
        'round': round_idx + 1, 'val_quarter': val_q,
        'roc_auc': auc, 'pr_auc': pr_auc, 'best_thresh': best_thresh, 'best_f1': best_f1,
        'precision': prec, 'recall': rec, 'spearman': spearman,
        'brier_raw': brier_raw, 'brier_calibrated': brier_calibrated,
    })

    val_oof = val_df[[ID_COL, QUARTER_COL, TARGET_COL]].copy()
    val_oof['pred_proba_raw'] = y_pred_proba_raw
    val_oof['pred_proba'] = y_pred_proba
    val_oof['pred_binary'] = y_pred_binary
    oof_predictions.append(val_oof)

    del train_df, val_df, X_train, y_train, X_val, y_val, inner_val_df, X_inner_val, y_inner_val
    gc.collect()

results_df = pd.DataFrame(round_results)
results_df.to_csv(ROUND_METRICS_PATH, index=False, encoding='utf-8-sig')
oof_df = pd.concat(oof_predictions, ignore_index=True)
oof_df.to_csv(OOF_PRED_PATH, index=False, encoding='utf-8-sig')

print('\n📈 [전체 라운드 평균 성능]')
print(results_df[['roc_auc', 'pr_auc', 'best_f1', 'precision', 'recall', 'spearman', 'brier_raw', 'brier_calibrated']].mean())

## 7. 캘리브레이션 확인

In [20]:
oof_df['pred_decile'] = pd.qcut(oof_df['pred_proba'], 10, labels=False, duplicates='drop')
calib = oof_df.groupby('pred_decile').agg(
    보정전_예측확률=('pred_proba_raw', 'mean'),
    보정후_예측확률=('pred_proba', 'mean'),
    실제폐업률=(TARGET_COL, 'mean'),
    건수=(TARGET_COL, 'size'),
)
print(calib)


             보정전_예측확률  보정후_예측확률     실제폐업률      건수
pred_decile                                      
0            0.210365  0.009125  0.011340  202293
1            0.257374  0.013941  0.017518  198195
2            0.285100  0.016957  0.030444  220605
3            0.313533  0.020458  0.035802  188286
4            0.350918  0.023254  0.052060  184076
5            0.400861  0.028749  0.044623  206080
6            0.442696  0.034929  0.037383  188027
7            0.497399  0.041898  0.044773  200457
8            0.552827  0.051496  0.063877  196721
9            0.628931  0.090686  0.111124  196421


In [22]:
import joblib
import os

SAVE_DIR = os.path.join(CHECKPOINT_DIR, 'saved_models_v6')
os.makedirs(SAVE_DIR, exist_ok=True)

X_full = df[FEATURE_COLS]
y_full = df[TARGET_COL].values   

final_model = LGBMClassifier(**FIXED_PARAMS)
final_model.fit(X_full, y_full)



[LightGBM] [Info] Number of positive: 263962, number of negative: 5777129
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.515414 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7295
[LightGBM] [Info] Number of data points in the train set: 6041091, number of used features: 48
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


LGBMClassifier(class_weight='balanced', colsample_bytree=0.8,
               learning_rate=0.03, max_depth=6, min_child_samples=100,
               n_estimators=300, n_jobs=-1, random_state=42, subsample=0.8)

In [23]:
last_q = ALL_QUARTERS[-1]

train_final_df = df[df[QUARTER_COL] != last_q]
calib_df = df[df[QUARTER_COL] == last_q]

X_full = train_final_df[FEATURE_COLS]
y_full = train_final_df[TARGET_COL].values

X_calib = calib_df[FEATURE_COLS]
y_calib = calib_df[TARGET_COL].values

final_model = LGBMClassifier(**FIXED_PARAMS)
final_model.fit(X_full, y_full)

final_calibrator = CalibratedClassifierCV(
    final_model,
    method='isotonic',
    cv='prefit'
)

final_calibrator.fit(
    X_calib,
    y_calib,
)

[LightGBM] [Info] Number of positive: 233283, number of negative: 5254273
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.509891 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7252
[LightGBM] [Info] Number of data points in the train set: 5487556, number of used features: 48
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


CalibratedClassifierCV(cv='prefit',
                       estimator=LGBMClassifier(class_weight='balanced',
                                                colsample_bytree=0.8,
                                                learning_rate=0.03, max_depth=6,
                                                min_child_samples=100,
                                                n_estimators=300, n_jobs=-1,
                                                random_state=42,
                                                subsample=0.8),
                       method='isotonic')